In [ ]:
# --- LOAD THE MAIN CLUSTERS ---

clustering_folder = '%s/clustering/res_%s' % (scvi_res_folder, str(cluster_resolution))
cluster_df_fileout = '%s/clusters.tsv' % (clustering_folder)
cluster_df = pd.read_table(cluster_df_fileout,sep='\t',index_col='index')
cluster_df = cluster_df.loc[list(adata.obs.index)]
adata.obs["cluster"] = np.array(cluster_df['leiden_scvi_cluster'])




In [ ]:
# --- GET THE SMC'S ---

smc_adata = adata[adata.obs['cluster'] == 0]
smc_adata

In [ ]:

# ** params **
min_cell_count = 100
min_marker_prop = 1e-4
transformed_min_prop = 1e-6
kernel_bandwidth = 0.1
std_residual_threshold = 0.5

# ** filter candidate marker based on pseudobulk **
pseudobulk_threshold = min_cell_count * min_marker_prop * np.median(smc_adata.obs['lib_size'])
pseudobulk_counts = np.array(np.sum(smc_adata.X,axis=0)).flatten() # [num_genes]
smc_adata.var['pseudobulk_counts'] = pseudobulk_counts
smc_adata = smc_adata[:,smc_adata.var['pseudobulk_counts'] >= pseudobulk_threshold]

# ** get transformed counts
X_transformed = smc_adata.X / np.array(smc_adata.obs['lib_size']).reshape(-1,1)
X_transformed = np.array(X_transformed)
X_transformed[X_transformed < transformed_min_prop] = transformed_min_prop
X_transformed = np.log10(X_transformed)




In [ ]:

# ** fit gaussian kernel to transformed counts mean / var **
import statsmodels
from statsmodels import nonparametric
from statsmodels.nonparametric import kernel_regression
X_transformed_mean = np.mean(X_transformed,axis=0)
X_transformed_var = np.var(X_transformed,axis=0)
model = statsmodels.nonparametric.kernel_regression.KernelReg(X_transformed_var, X_transformed_mean.reshape(-1,1), var_type = ['c'], bw = [kernel_bandwidth]) 
pred_var,marginal_effects = model.fit()





In [ ]:
# ** compute the pearson residuals **
est_std = (np.mean((X_transformed_var - pred_var)**2)) ** 0.5
pearson_residuals = (X_transformed_var - pred_var) / est_std
hv_gene_indices = np.where(pearson_residuals >= std_residual_threshold)[0]






In [ ]:
# ** viz **
plt.clf()
plt.scatter(X_transformed_mean,X_transformed_var,s=1,c='b')
plt.scatter(X_transformed_mean[hv_gene_indices],X_transformed_var[hv_gene_indices],s=1,c='r')
plt.title("Transformed count var vs. mean colored by HV indicator")
plt.xlabel("Transformed count mean")
plt.ylabel("Transformed count var")
# plt.savefig("/users/benauerbach/desktop/hv_fig.png",dpi=200)
plt.show()




In [ ]:
# ** get hv gene names **
hv_genes = np.array(smc_adata.var_names[hv_gene_indices])


